# Adaptive Sentiment Orchestration (ASO) + SPIT
### Hybrid Framework with Dynamic Socio-Physical Thresholding

---
**Paper:** *Adaptive Sentiment Orchestration via Socio-Physical Integrated Thresholding (SPIT) for High-Velocity Social Streams*

**Architecture:**
```
Input Text
    |
    v
[ Tier-1: DistilBERT ] --> confidence c + token_ids
    |
    v
[ SPIT: Dynamic Threshold (NOVEL) ]
  Hardware Telemetry Sidecar (GPU *or* CPU — real sensors, no simulation)
    cpu_load, ram, cpu_temp, swap  ←  psutil  (always)
    gpu_temp, vram                 ←  pynvml  (when GPU present)
  PID --> Phi_physical
  Socio Signals --> Psi_socio
  Token IDs     --> H_token
  tau = sigmoid(Phi*(1-Psi)+H)*(tau_max-tau_min)+tau_min
    |
  conf >= tau_dynamic?
  YES --> DistilBERT (Tier-1)  |  NO --> BERT (Tier-2)
```

**5 Models evaluated:**
1. Logistic Regression (TF-IDF baseline)
2. DistilBERT (Tier-1 only)
3. BERT (Tier-2 only)
4. ASO Hybrid — fixed tau=0.85
5. **ASO+SPIT** — dynamic tau (this paper)

---
> **Device:** Runs on **GPU** (T4 recommended) **or CPU**. Pipeline auto-detects hardware and adapts batch size and telemetry accordingly.  
> Colab: Runtime → Change runtime type → T4 GPU *(optional, CPU is fully supported)*

## Cell 1 — Install Dependencies

In [ ]:
# ============================================================
# CELL 1: Install Dependencies  (run once per Colab session)
# ============================================================
!pip install -q transformers datasets scikit-learn torch pandas matplotlib
!pip install -q accelerate sentencepiece pynvml psutil
print("All dependencies installed.")

## Cell 2 — Configuration & Imports

In [ ]:
# ============================================================
# CELL 2: Global Configuration & Imports
# ============================================================
import os, sys, logging, warnings, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device Detection ────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ON_GPU = DEVICE == 'cuda'

print('=' * 58)
print('  HARDWARE ENVIRONMENT')
print('=' * 58)
print(f'  PyTorch version : {torch.__version__}')
print(f'  Device          : {DEVICE.upper()}')

if ON_GPU:
    _gp = torch.cuda.get_device_properties(0)
    print(f'  GPU Name        : {_gp.name}')
    print(f'  GPU VRAM        : {_gp.total_memory/1024**3:.1f} GB')
    print(f'  CUDA version    : {torch.version.cuda}')
else:
    try:
        import psutil as _ps
        _cpu_phys    = _ps.cpu_count(logical=False)
        _cpu_logical = _ps.cpu_count(logical=True)
        _ram_gb      = _ps.virtual_memory().total / 1024**3
        print(f'  CPU             : {platform.processor() or platform.machine()}')
        print(f'  CPU Cores       : {_cpu_phys} physical / {_cpu_logical} logical')
        print(f'  System RAM      : {_ram_gb:.1f} GB')
    except ImportError:
        print('  [psutil not installed yet — run Cell 1 first]')
    print('  NOTE: Running on CPU — inference is slower but fully functional.')
    print('  NOTE: SPIT uses real CPU telemetry (psutil) — no simulation.')
print('=' * 58)

# Dataset
DATA_SOURCE   = 'sst2'
MAX_SAMPLES   = 4000
TEST_SIZE     = 0.20
CSV_PATH      = None

# Models
TIER1_MODEL   = 'distilbert-base-uncased-finetuned-sst-2-english'
TIER2_MODEL   = 'textattack/bert-base-uncased-SST-2'

# ASO (fixed threshold)
ASO_THRESHOLD = 0.85

# SPIT configuration
SPIT_T_SETPOINT     = 85.0   # GPU thermal set-point °C
SPIT_T_CPU_SETPOINT = 90.0   # CPU thermal set-point °C
SPIT_VRAM_GB        = 16.0   # Colab T4 VRAM
SPIT_KP             = 0.008
SPIT_KI             = 0.002
SPIT_KD             = 0.004
SPIT_TAU_MIN        = 0.50
SPIT_TAU_MAX        = 0.92

# Auto-detect system RAM for CPU mode
try:
    import psutil as _ps2
    SPIT_RAM_GB = _ps2.virtual_memory().total / 1024**3
except Exception:
    SPIT_RAM_GB = 12.0   # Colab default

# Adaptive inference batch size
INFERENCE_BATCH_SIZE = 32 if ON_GPU else 8

print(f'\n  Tier-1 model       : {TIER1_MODEL}')
print(f'  Tier-2 model       : {TIER2_MODEL}')
print(f'  ASO tau (fixed)    : {ASO_THRESHOLD}')
print(f'  SPIT tau range     : [{SPIT_TAU_MIN}, {SPIT_TAU_MAX}]')
print(f'  SPIT T_setpoint    : {SPIT_T_SETPOINT} °C (GPU) / {SPIT_T_CPU_SETPOINT} °C (CPU)')
print(f'  RAM detected       : {SPIT_RAM_GB:.1f} GB')
print(f'  Inference batch    : {INFERENCE_BATCH_SIZE} samples/batch')

## Cell 3 — Load & Preprocess Dataset

In [ ]:
# ============================================================
# CELL 3: Dataset Loading & Preprocessing
# Uses setup_and_data.py (or inline if running standalone)
# ============================================================

# ── If running inside the Research paper directory: ──────────
# sys.path.insert(0, '/content/drive/MyDrive/ASO/')  # Colab Drive path
# from setup_and_data import get_data

# ── Inline implementation for standalone Colab use ───────────
import re
from datasets import load_dataset
from sklearn.model_selection import train_test_split


def clean_text(text: str) -> str:
    """Normalise text: lower-case, strip URLs, handles, special chars."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_dataset_split(source, max_samples, test_size, seed, csv_path=None):
    if source == "sst2":
        ds = load_dataset("glue", "sst2", split="train")
        texts  = list(ds["sentence"])
        labels = list(ds["label"])
    elif source == "tweet_eval":
        ds = load_dataset("tweet_eval", "sentiment", split="train")
        texts, labels = [], []
        for t, l in zip(ds["text"], ds["label"]):
            if l == 0:   texts.append(t); labels.append(0)
            elif l == 2: texts.append(t); labels.append(1)
    elif source == "sentiment140":
        import pandas as _pd
        df = _pd.read_csv(csv_path, encoding="latin-1", header=None,
                          usecols=[0, 5], names=["polarity", "text"])
        texts  = df["text"].tolist()
        labels = (df["polarity"] == 4).astype(int).tolist()
    else:
        raise ValueError(f"Unknown source: {source}")

    if max_samples and max_samples < len(texts):
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(texts), size=max_samples, replace=False)
        texts  = [texts[i]  for i in idx]
        labels = [labels[i] for i in idx]

    texts = [clean_text(t) for t in texts]
    return train_test_split(texts, labels, test_size=test_size,
                            random_state=seed, stratify=labels)


print("Loading dataset …")
X_train, X_test, y_train, y_test = load_dataset_split(
    DATA_SOURCE, MAX_SAMPLES, TEST_SIZE, SEED, CSV_PATH
)

print(f"\nDataset: {DATA_SOURCE}")
print(f"Train samples : {len(X_train)}")
print(f"Test  samples : {len(X_test)}")
print(f"Positive ratio (test): {np.mean(y_test):.2%}")
print("\nSample texts:")
for i in range(3):
    label_str = "POS" if y_test[i] == 1 else "NEG"
    print(f"  [{label_str}] {X_test[i][:80]}…")

## Cell 4 — Baseline: Logistic Regression (TF-IDF)

In [ ]:
# ============================================================
# CELL 4: Logistic Regression Baseline
# Classic NLP benchmark using TF-IDF features.
# Provides a strong non-neural reference point.
# ============================================================

import time
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

print("Training Logistic Regression …")
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2,
        strip_accents="unicode",
        analyzer="word",
        token_pattern=r"\w{1,}",
    )),
    ("lr", LogisticRegression(
        C=1.0, max_iter=1000, solver="saga", n_jobs=-1
    )),
])

t0 = time.perf_counter()
lr_pipeline.fit(X_train, y_train)
train_time = time.perf_counter() - t0

# ── Inference with per-sample latency ─────────────────────────
latencies = []
all_preds = []
BATCH = 64
for start in range(0, len(X_test), BATCH):
    batch = X_test[start : start + BATCH]
    t0 = time.perf_counter()
    preds = lr_pipeline.predict(batch)
    elapsed = time.perf_counter() - t0
    latencies.extend([elapsed / len(batch)] * len(batch))
    all_preds.extend(preds)

# ── Metrics ────────────────────────────────────────────────────
lr_acc  = accuracy_score(y_test, all_preds)
lr_f1   = f1_score(y_test, all_preds, average="macro", zero_division=0)
lr_lat  = float(np.mean(latencies)) * 1000   # ms
lr_preds = all_preds

print(f"Training time : {train_time:.2f}s")
print(f"Accuracy      : {lr_acc:.4f}")
print(f"F1 (Macro)    : {lr_f1:.4f}")
print(f"Avg latency   : {lr_lat:.4f} ms/sample")

## Cell 5 — Tier-1: DistilBERT

In [ ]:
# ============================================================
# CELL 5: Tier-1 — DistilBERT Inference
# Pre-trained and fine-tuned on SST-2; no additional training
# required. Fast inference (~60% faster than BERT).
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

print(f"Loading Tier-1 model: {TIER1_MODEL} …")
t1_tokenizer = AutoTokenizer.from_pretrained(TIER1_MODEL)
t1_model     = AutoModelForSequenceClassification.from_pretrained(TIER1_MODEL)
t1_model.to(DEVICE).eval()
print("Tier-1 model loaded.")

# Label map: DistilBERT SST-2 → {0: NEGATIVE, 1: POSITIVE}
T1_LABEL_MAP = {0: 0, 1: 1}   # Model id 0 = NEG, 1 = POS


@torch.no_grad()
def t1_predict_batch(texts):
    """Return (predictions, confidences) for a list of texts."""
    enc = t1_tokenizer(
        texts, padding=True, truncation=True,
        max_length=128, return_tensors="pt"
    ).to(DEVICE)
    logits = t1_model(**enc).logits
    probs  = F.softmax(logits, dim=-1).cpu().numpy()
    raw    = np.argmax(probs, axis=-1)
    preds  = [T1_LABEL_MAP[int(p)] for p in raw]
    confs  = [float(probs[i, raw[i]]) for i in range(len(texts))]
    return preds, confs


# ── Warm-up (discard first batch latency to avoid JIT overhead) ─
_ = t1_predict_batch(X_test[:INFERENCE_BATCH_SIZE])

# ── Full test set inference with latency tracking ───────────────
t1_preds, t1_confs, t1_latencies = [], [], []

for start in range(0, len(X_test), INFERENCE_BATCH_SIZE):
    batch = X_test[start : start + INFERENCE_BATCH_SIZE]
    t0 = time.perf_counter()
    preds, confs = t1_predict_batch(batch)
    elapsed = time.perf_counter() - t0
    lat_per_sample = elapsed / len(batch)
    t1_preds.extend(preds)
    t1_confs.extend(confs)
    t1_latencies.extend([lat_per_sample] * len(batch))

# ── Metrics ─────────────────────────────────────────────────────
t1_acc = accuracy_score(y_test, t1_preds)
t1_f1  = f1_score(y_test, t1_preds, average="macro", zero_division=0)
t1_lat = float(np.mean(t1_latencies)) * 1000

print(f"\nTier-1 (DistilBERT) Results:")
print(f"  Accuracy      : {t1_acc:.4f}")
print(f"  F1 (Macro)    : {t1_f1:.4f}")
print(f"  Avg latency   : {t1_lat:.4f} ms/sample")
print(f"  Avg confidence: {np.mean(t1_confs):.4f}")
print(f"  Samples below threshold ({ASO_THRESHOLD}): "
      f"{sum(1 for c in t1_confs if c < ASO_THRESHOLD)} / {len(t1_confs)} "
      f"({sum(1 for c in t1_confs if c < ASO_THRESHOLD)/len(t1_confs):.1%})")

## Cell 6 — Tier-2: BERT

In [ ]:
# ============================================================
# CELL 6: Tier-2 — BERT Inference
# Stronger model for ambiguous samples.
# Evaluated standalone to establish upper-bound performance.
# ============================================================

print(f"Loading Tier-2 model: {TIER2_MODEL} …")
t2_tokenizer = AutoTokenizer.from_pretrained(TIER2_MODEL)
t2_model     = AutoModelForSequenceClassification.from_pretrained(TIER2_MODEL)
t2_model.to(DEVICE).eval()
print("Tier-2 model loaded.")

# Auto-detect label map from model config
id2label = t2_model.config.id2label
print(f"Model label map: {id2label}")

def _make_label_map(id2label):
    mapping = {}
    for idx, name in id2label.items():
        nl = name.lower()
        if "neg" in nl or nl in ("label_0", "0", "false"):
            mapping[idx] = 0
        elif "pos" in nl or nl in ("label_1", "1", "true"):
            mapping[idx] = 1
        else:
            mapping[idx] = 0 if idx == min(id2label.keys()) else 1
    return mapping

T2_LABEL_MAP = _make_label_map(id2label)
print(f"Resolved label map: {T2_LABEL_MAP}")


@torch.no_grad()
def t2_predict_batch(texts):
    """Return (predictions, confidences) for a list of texts."""
    enc = t2_tokenizer(
        texts, padding=True, truncation=True,
        max_length=256, return_tensors="pt"
    ).to(DEVICE)
    logits = t2_model(**enc).logits
    probs  = F.softmax(logits, dim=-1).cpu().numpy()
    raw    = np.argmax(probs, axis=-1)
    preds  = [T2_LABEL_MAP[int(p)] for p in raw]
    confs  = [float(probs[i, raw[i]]) for i in range(len(texts))]
    return preds, confs


# ── Warm-up ─────────────────────────────────────────────────────
_ = t2_predict_batch(X_test[:INFERENCE_BATCH_SIZE])

# ── Full inference ───────────────────────────────────────────────
t2_preds_all, t2_latencies = [], []

for start in range(0, len(X_test), INFERENCE_BATCH_SIZE):
    batch = X_test[start : start + INFERENCE_BATCH_SIZE]
    t0 = time.perf_counter()
    preds, _ = t2_predict_batch(batch)
    elapsed = time.perf_counter() - t0
    t2_preds_all.extend(preds)
    t2_latencies.extend([elapsed / len(batch)] * len(batch))

t2_acc = accuracy_score(y_test, t2_preds_all)
t2_f1  = f1_score(y_test, t2_preds_all, average="macro", zero_division=0)
t2_lat = float(np.mean(t2_latencies)) * 1000

print(f"\nTier-2 (BERT) Results:")
print(f"  Accuracy      : {t2_acc:.4f}")
print(f"  F1 (Macro)    : {t2_f1:.4f}")
print(f"  Avg latency   : {t2_lat:.4f} ms/sample")

## Cell 7 — ASO: Adaptive Router

In [ ]:
# ============================================================
# CELL 7: ASO — Adaptive Routing Inference
# Core contribution: route samples by Tier-1 confidence.
# Samples below threshold τ are escalated to Tier-2.
# ============================================================

print(f"Running ASO with threshold τ = {ASO_THRESHOLD} …")

aso_preds     = []
aso_latencies = []
tier1_routed  = 0
tier2_routed  = 0

total_t0 = time.perf_counter()

for start in range(0, len(X_test), INFERENCE_BATCH_SIZE):
    batch = X_test[start : start + INFERENCE_BATCH_SIZE]

    # ── Step 1: Tier-1 inference ──────────────────────────────
    t0 = time.perf_counter()
    t1_batch_preds, t1_batch_confs = t1_predict_batch(batch)
    t1_elapsed = time.perf_counter() - t0
    t1_per_sample = t1_elapsed / len(batch)

    # ── Step 2: Partition by confidence ──────────────────────
    confident_mask   = [c >= ASO_THRESHOLD for c in t1_batch_confs]
    escalate_indices = [i for i, m in enumerate(confident_mask) if not m]
    escalate_texts   = [batch[i] for i in escalate_indices]

    final_preds = list(t1_batch_preds)  # start with Tier-1 predictions
    tier1_routed += sum(confident_mask)
    tier2_routed += len(escalate_indices)

    # ── Step 3: Escalate low-confidence samples ───────────────
    t2_elapsed = 0.0
    t2_per_sample = 0.0
    if escalate_texts:
        t0 = time.perf_counter()
        t2_batch_preds, _ = t2_predict_batch(escalate_texts)
        t2_elapsed = time.perf_counter() - t0
        t2_per_sample = t2_elapsed / len(escalate_texts)
        # Overwrite with Tier-2 predictions
        for rank, idx in enumerate(escalate_indices):
            final_preds[idx] = t2_batch_preds[rank]

    # ── Step 4: Compute per-sample latency ───────────────────
    for i in range(len(batch)):
        if confident_mask[i]:
            # Tier-1 only: proportional share of t1 batch time
            aso_latencies.append(t1_per_sample)
        else:
            # Tier-1 + Tier-2: both tiers ran
            aso_latencies.append(t1_per_sample + t2_per_sample)

    aso_preds.extend(final_preds)

total_aso_time = time.perf_counter() - total_t0

aso_acc = accuracy_score(y_test, aso_preds)
aso_f1  = f1_score(y_test, aso_preds, average="macro", zero_division=0)
aso_lat = float(np.mean(aso_latencies)) * 1000

total_samples  = len(X_test)
tier2_rate     = tier2_routed / total_samples

print(f"\nASO Results (τ = {ASO_THRESHOLD}):")
print(f"  Accuracy        : {aso_acc:.4f}")
print(f"  F1 (Macro)      : {aso_f1:.4f}")
print(f"  Avg latency     : {aso_lat:.4f} ms/sample")
print(f"  Total time      : {total_aso_time:.3f}s")
print(f"  Tier-1 answered : {tier1_routed}/{total_samples} ({1-tier2_rate:.1%})")
print(f"  Tier-2 invoked  : {tier2_routed}/{total_samples} ({tier2_rate:.1%})")

## Cell 7b ? SPIT: Socio-Physical Integrated Thresholding (Novel Contribution)

SPIT replaces the static threshold with a **per-sample dynamic threshold** computed from three signal families:

| Signal | Symbol | Source | Effect on tau |
|---|---|---|---|
| GPU thermal pressure | `Phi_physical` | PID on live GPU temp | Hot GPU -> raise tau (frugal) |
| Socio-impact multiplier | `Psi_socio` | Virality + authority + burst + platform | Urgent post -> lower tau (escalate) |
| Linguistic entropy | `H_token` | Token-ID entropy | Complex text -> lower tau (escalate) |

**Formula:** `tau_dynamic = sigmoid(Phi*(1-Psi) + H) * (tau_max - tau_min) + tau_min`

In [ ]:
# ============================================================
# CELL 7c: SPIT Module — §1-§7 Production Implementation
# GPU + CPU aware | real hardware telemetry | no simulation
# ============================================================
import numpy as np
import threading
import time as _time
import warnings as _warnings
from dataclasses import dataclass as _dc, field as _field
from typing import Optional as _Opt
from enum import Enum as _Enum

# ── psutil (CPU telemetry — always first) ────────────────────
try:
    import psutil as _psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    PSUTIL_AVAILABLE = False
    _warnings.warn(
        "[SPIT] psutil not available. CPU telemetry will be zeroed. "
        "Install with: !pip install psutil",
        RuntimeWarning
    )

# ── pynvml (GPU telemetry — only when GPU present) ───────────
try:
    import pynvml as _pynvml
    _pynvml.nvmlInit()
    NVML_AVAILABLE = True
except Exception:
    NVML_AVAILABLE = False

# ── Announce telemetry mode ───────────────────────────────────
if NVML_AVAILABLE:
    _TELEMETRY_MODE = "GPU+CPU"
elif PSUTIL_AVAILABLE:
    _TELEMETRY_MODE = "CPU"
else:
    _TELEMETRY_MODE = "UNAVAILABLE"

print(f"[SPIT] Telemetry mode: {_TELEMETRY_MODE}")


# =============================================================
# §1  Enumerations & Config
# =============================================================

class PlatformType(_Enum):
    """Platform-level socio-entropy H_socio. Higher → more ambiguity → lower tau."""
    TWITTER   = 0.35
    REDDIT    = 0.45
    INSTAGRAM = 0.60
    TIKTOK    = 0.75
    UNKNOWN   = 0.50

@_dc
class SPITConfig:
    # Physical set-points
    T_setpoint:       float = 85.0     # GPU thermal set-point (°C)
    T_cpu_setpoint:   float = 90.0     # CPU thermal set-point (°C)
    T_min:            float = 40.0     # Idle temperature floor (°C)
    VRAM_capacity:    float = 16.0     # Total VRAM (GB)
    RAM_capacity_gb:  float = 12.0     # Total system RAM (GB)
    # PID gains
    Kp: float = 0.008
    Ki: float = 0.002
    Kd: float = 0.004
    # Threshold bounds
    tau_min: float = 0.50
    tau_max: float = 0.92
    # Socio caps
    retweet_cap:      float = 1_000_000.0
    follower_cap:     float = 10_000_000.0
    burst_cap:        float = 500.0
    # Sidecar
    poll_interval_ms: float = 20.0
    integral_limit:   float = 50.0


# =============================================================
# §2  Hardware Telemetry Sidecar
# =============================================================

class HardwareTelemetrySidecar:
    """
    Background thread polling real hardware at poll_interval_ms cadence.
    Exposes a lock-free 6-element health vector:

      h[0] cpu_load_norm    (psutil, always)
      h[1] ram_norm         (psutil, always)
      h[2] cpu_temp_norm    (psutil sensors — 0.0 if VM doesn't expose them)
      h[3] swap_norm        (psutil, always)
      h[4] gpu_temp_norm    (pynvml  — 0.0 if no GPU)
      h[5] vram_norm        (pynvml  — 0.0 if no GPU)

    No simulation. Values are real hardware readings.
    """

    VECTOR_LABELS = [
        "cpu_load_norm", "ram_usage_norm", "cpu_temp_norm",
        "swap_norm", "gpu_temp_norm", "vram_usage_norm",
    ]

    def __init__(self, config: SPITConfig):
        self.config = config
        self._lock  = threading.Lock()
        self._health_vector = np.zeros(6, dtype=np.float32)
        self._running = False
        self._thread: _Opt[threading.Thread] = None
        self._gpu_handle = None
        if NVML_AVAILABLE:
            try:
                self._gpu_handle = _pynvml.nvmlDeviceGetHandleByIndex(0)
            except Exception:
                pass
        self._total_ram_gb = (
            _psutil.virtual_memory().total / (1024 ** 3)
            if PSUTIL_AVAILABLE else config.RAM_capacity_gb
        )

    def start(self):
        h = self._read_hardware()
        with self._lock:
            self._health_vector = h
        self._running = True
        self._thread = threading.Thread(
            target=self._poll_loop, daemon=True, name="SPIT-Telemetry"
        )
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(timeout=1.0)

    def read(self) -> np.ndarray:
        with self._lock:
            return self._health_vector.copy()

    def read_labeled(self) -> dict:
        h = self.read()
        return {lbl: float(h[i]) for i, lbl in enumerate(self.VECTOR_LABELS)}

    def _poll_loop(self):
        iv = self.config.poll_interval_ms / 1000.0
        while self._running:
            h = self._read_hardware()
            with self._lock:
                self._health_vector = h
            _time.sleep(iv)

    def _read_hardware(self) -> np.ndarray:
        return np.concatenate([self._read_cpu(), self._read_gpu()]).astype(np.float32)

    def _read_cpu(self) -> np.ndarray:
        if not PSUTIL_AVAILABLE:
            return np.zeros(4, dtype=np.float32)
        try:
            cpu_load_norm = np.clip(_psutil.cpu_percent(interval=None) / 100.0, 0.0, 1.0)
            vm  = _psutil.virtual_memory()
            ram_norm = np.clip(vm.used / vm.total, 0.0, 1.0)
            cpu_temp_norm = self._read_cpu_temp()
            swap = _psutil.swap_memory()
            swap_norm = np.clip(swap.used / swap.total, 0.0, 1.0) if swap.total > 0 else 0.0
            return np.array([cpu_load_norm, ram_norm, cpu_temp_norm, swap_norm], dtype=np.float32)
        except Exception:
            return np.zeros(4, dtype=np.float32)

    def _read_cpu_temp(self) -> float:
        try:
            sensors = _psutil.sensors_temperatures()
            if not sensors:
                return 0.0
            for key in ("coretemp", "k10temp", "cpu_thermal", "acpitz", "cpu-thermal"):
                if key in sensors:
                    temps = [r.current for r in sensors[key] if r.current is not None]
                    if temps:
                        return float(np.clip(max(temps) / self.config.T_cpu_setpoint, 0.0, 1.0))
            for entries in sensors.values():
                temps = [r.current for r in entries if r.current is not None]
                if temps:
                    return float(np.clip(max(temps) / self.config.T_cpu_setpoint, 0.0, 1.0))
        except Exception:
            pass
        return 0.0

    def _read_gpu(self) -> np.ndarray:
        if not (NVML_AVAILABLE and self._gpu_handle is not None):
            return np.zeros(2, dtype=np.float32)
        try:
            temp_c    = _pynvml.nvmlDeviceGetTemperature(self._gpu_handle, _pynvml.NVML_TEMPERATURE_GPU)
            mem_info  = _pynvml.nvmlDeviceGetMemoryInfo(self._gpu_handle)
            vram_gb   = mem_info.used / (1024 ** 3)
            return np.array([
                np.clip(temp_c / self.config.T_setpoint, 0.0, 1.0),
                np.clip(vram_gb / self.config.VRAM_capacity, 0.0, 1.0),
            ], dtype=np.float32)
        except Exception:
            return np.zeros(2, dtype=np.float32)


# =============================================================
# §3  PID Controller  (CPU-aware)
# =============================================================

class PIDController:
    """
    Maps the 6-element health vector → Φ_physical ∈ [0, 1].

    GPU mode:  primary error = GPU temperature vs T_setpoint (°C)
    CPU mode:  primary error = CPU load re-expressed as pseudo-temperature
               so that the same PID gains remain numerically valid.
               CPU temperature is added as secondary when exposed by the VM.

    Φ_physical → 0  hardware relaxed (threshold can be lower / more diligent)
    Φ_physical → 1  hardware stressed (threshold raised / more frugal)
    """
    def __init__(self, config: SPITConfig):
        self.Kp = config.Kp; self.Ki = config.Ki; self.Kd = config.Kd
        self.T_setpoint     = config.T_setpoint
        self.T_cpu_setpoint = config.T_cpu_setpoint
        self.integral_limit = config.integral_limit
        self._integral: float = 0.0
        self._prev_error: float = 0.0
        self._prev_time:  float = _time.time()

    def step(self, health_vector: np.ndarray) -> float:
        """
        health_vector shape (6,):
            [cpu_load, ram, cpu_temp, swap, gpu_temp, vram]
        """
        now = _time.time()
        dt  = max(now - self._prev_time, 1e-6)

        cpu_load_norm = float(health_vector[0])
        ram_norm      = float(health_vector[1])
        cpu_temp_norm = float(health_vector[2])
        swap_norm     = float(health_vector[3])
        gpu_temp_norm = float(health_vector[4])
        vram_norm     = float(health_vector[5])

        gpu_present = gpu_temp_norm > 0.0

        if gpu_present:
            T_gpu    = gpu_temp_norm * self.T_setpoint
            error    = T_gpu - self.T_setpoint
            secondary = 0.08 * vram_norm + 0.04 * ram_norm
        else:
            # CPU pseudo-thermal mapping: 100% load ≡ T_cpu_setpoint
            T_cpu_equiv = cpu_load_norm * self.T_cpu_setpoint
            error       = T_cpu_equiv - self.T_cpu_setpoint
            cpu_temp_contrib = cpu_temp_norm * self.T_cpu_setpoint if cpu_temp_norm > 0 else 0.0
            secondary = (
                0.08 * ram_norm
                + 0.05 * swap_norm
                + 0.04 * (cpu_temp_contrib / self.T_cpu_setpoint)
            )

        P = self.Kp * error
        self._integral = np.clip(
            self._integral + error * dt, -self.integral_limit, self.integral_limit
        )
        I = self.Ki * self._integral
        D = self.Kd * (error - self._prev_error) / dt

        self._prev_error = error
        self._prev_time  = now
        return float(np.clip(P + I + D + secondary, 0.0, 1.0))


# =============================================================
# §4  Ψ_socio
# =============================================================

def compute_psi_socio(rt, fc, bpm, plat, cfg):
    V = np.clip(rt / cfg.retweet_cap, 0.0, 1.0)
    A = (np.clip(np.log1p(fc) / np.log1p(cfg.follower_cap), 0.0, 1.0) if fc > 0 else 0.0)
    B = np.clip(bpm / cfg.burst_cap, 0.0, 1.0)
    H = plat.value
    return float(np.clip(np.dot([0.30, 0.30, 0.25, 0.15], [V, A, B, H]), 0.0, 1.0))


# =============================================================
# §5  H_token
# =============================================================

def compute_h_token(ids, vsz=30522):
    if not ids:
        return 0.0
    _, c = np.unique(np.array(ids, dtype=np.int32), return_counts=True)
    p = c / c.sum()
    return float(np.clip(-np.sum(p * np.log(p + 1e-12)) / np.log(vsz + 1e-12), 0.0, 1.0))


# =============================================================
# §6  SPIT Function
# =============================================================

def spit(phi_physical, retweet_count, follower_count, burst_rate_ppm,
         platform, token_ids, config=None):
    """
    τ_dynamic = σ( Φ_physical·(1−Ψ_socio) + H_token ) · (τ_max−τ_min) + τ_min

    Returns dict of all components (for logging/ablation).
    """
    if config is None:
        config = SPITConfig()
    psi   = compute_psi_socio(retweet_count, follower_count, burst_rate_ppm, platform, config)
    h     = compute_h_token(token_ids)
    pt    = phi_physical * (1.0 - psi)
    z     = pt + h
    sg    = 1.0 / (1.0 + np.exp(-z))
    tau   = float(np.clip(sg * (config.tau_max - config.tau_min) + config.tau_min,
                          config.tau_min, config.tau_max))
    return {
        "tau_dynamic":   tau,
        "phi_physical":  phi_physical,
        "psi_socio":     psi,
        "h_token":       h,
        "physical_term": pt,
        "sigmoid_input": z,
    }


# =============================================================
# §7  Data Classes & Cascade Router
# =============================================================

@_dc
class PostContext:
    text:            str
    token_ids:       list
    distilbert_conf: float
    retweet_count:   float = 0.0
    follower_count:  float = 0.0
    burst_rate_ppm:  float = 0.0
    platform:        PlatformType = PlatformType.UNKNOWN

@_dc
class RoutingDecision:
    escalate:          bool
    tau_dynamic:       float
    distilbert_conf:   float
    spit_components:   dict
    hardware_snapshot: dict           # labeled telemetry at decision time
    tier: str = _field(init=False)
    def __post_init__(self):
        self.tier = "BERT (Tier-2)" if self.escalate else "DistilBERT (Tier-1)"

class SPITCascadeRouter:
    def __init__(self, config=None):
        if config is None:
            config = SPITConfig()
        self.config  = config
        self.sidecar = HardwareTelemetrySidecar(config)
        self.pid     = PIDController(config)
        self._started = False

    def start(self):
        self.sidecar.start()
        self._started = True
        print("[SPIT] Cascade router started. Telemetry sidecar active.")

    def stop(self):
        self.sidecar.stop()
        print("[SPIT] Cascade router stopped.")

    def route(self, post: PostContext) -> RoutingDecision:
        if not self._started:
            raise RuntimeError("Call router.start() before routing posts.")
        h   = self.sidecar.read()          # shape (6,)
        phi = self.pid.step(h)
        r   = spit(phi, post.retweet_count, post.follower_count,
                   post.burst_rate_ppm, post.platform, post.token_ids, self.config)
        return RoutingDecision(
            escalate          = post.distilbert_conf < r["tau_dynamic"],
            tau_dynamic       = r["tau_dynamic"],
            distilbert_conf   = post.distilbert_conf,
            spit_components   = r,
            hardware_snapshot = self.sidecar.read_labeled(),
        )


print(f"[SPIT] Module loaded | Telemetry: {_TELEMETRY_MODE} | Device: {DEVICE.upper()}")


## Cell 7d ? Synthetic Social Metadata Generation

SST-2 contains movie-review text with no real social metadata.
We generate **reproducible synthetic context** per sample using Twitter-calibrated distributions:

| Signal | Distribution | Real-world calibration |
|---|---|---|
| `follower_count` | Log-normal (mu=8.5, sigma=2.0) | Median ~4,900; 95th-pct ~1.2M |
| `retweet_count` | Log-normal (mu=3.0, sigma=2.5) | Median ~20; 95th-pct ~4,000 |
| `burst_rate_ppm` | Uniform [1, 200] | Topic velocity range |
| `platform` | TWITTER | Consistent domain assumption |

Saved to `spit_synthetic_metadata.csv` for paper appendix.

In [ ]:
# ============================================================
# CELL 7e: Synthetic Social Metadata  (seeded, reproducible)
# ============================================================
np.random.seed(SEED)
N = len(X_test)

# Follower counts: heavy-tailed log-normal (Twitter distribution)
synth_followers = np.clip(np.random.lognormal(8.5, 2.0, N), 0, 10_000_000)

# Retweet counts: log-normal (most <10 RT, few go viral)
synth_retweets  = np.clip(np.random.lognormal(3.0, 2.5, N), 0, 1_000_000)

# Burst rate: uniform [1, 200] posts/min
synth_burst     = np.random.uniform(1.0, 200.0, N)

# Platform: all TWITTER for SST-2 domain
synth_platform_enum = PlatformType.TWITTER

synth_meta_df = pd.DataFrame({
    "follower_count": synth_followers.astype(int),
    "retweet_count":  synth_retweets.astype(int),
    "burst_rate_ppm": synth_burst.round(2),
    "platform":       ["TWITTER"] * N,
})

print("=" * 58)
print("  Synthetic Social Metadata -- Summary Statistics")
print("=" * 58)
print(synth_meta_df[["follower_count","retweet_count","burst_rate_ppm"]].describe().round(2).to_string())
print(f"\n  Platform  : {synth_meta_df['platform'].value_counts().to_dict()}")
print(f"  Total rows: {len(synth_meta_df)}")
print("=" * 58)

synth_meta_df.to_csv("spit_synthetic_metadata.csv", index=False)
print("\nSaved: spit_synthetic_metadata.csv")

## Cell 7f ? SPIT Adaptive Routing Inference

Steps:
1. Tokenise all test texts -> `all_token_ids` (for H_token computation)
2. Start `SPITCascadeRouter` (launches sidecar thread)
3. Per batch: read hardware -> step PID -> compute tau_dynamic per sample
4. Build escalation mask: `escalate[i] = t1_conf[i] < tau_dynamic[i]`
5. Batch Tier-2 call for escalated samples
6. Compute metrics

In [ ]:
# ============================================================
# CELL 7g: SPIT Inference on Full Test Set
# ============================================================
import time

# -- Build SPITConfig from global settings --
spit_config = SPITConfig(
    T_setpoint      = SPIT_T_SETPOINT,
    T_cpu_setpoint  = SPIT_T_CPU_SETPOINT,
    VRAM_capacity   = SPIT_VRAM_GB,
    RAM_capacity_gb = SPIT_RAM_GB,
    Kp=SPIT_KP, Ki=SPIT_KI, Kd=SPIT_KD,
    tau_min=SPIT_TAU_MIN, tau_max=SPIT_TAU_MAX,
)
print(f"[SPIT] Config: device={DEVICE.upper()}, "
      f"T_setpoint={spit_config.T_setpoint}°C (GPU) / "
      f"{spit_config.T_cpu_setpoint}°C (CPU), "
      f"resource_cap={'VRAM '+str(SPIT_VRAM_GB)+'GB' if ON_GPU else 'RAM '+str(round(SPIT_RAM_GB,1))+'GB'}")

# Step 1: Tokenise for H_token
print("[SPIT] Tokenising test set ...")
all_token_ids = [
    t1_tokenizer.encode(txt, truncation=True, max_length=128)
    for txt in X_test
]

# Step 2: Start router
spit_router = SPITCascadeRouter(spit_config)
spit_router.start()
time.sleep(0.1)   # let sidecar pre-read hardware

# Step 3: Compute tau_dynamic per sample via sidecar + PID
print("[SPIT] Computing dynamic thresholds ...")
spit_tau_vals=[]; spit_phi_vals=[]; spit_psi_vals=[]; spit_htoken_vals=[]
spit_hw_snapshots = []    # hardware telemetry at decision time (for ablation)

for start in range(0, len(X_test), INFERENCE_BATCH_SIZE):
    end   = min(start + INFERENCE_BATCH_SIZE, len(X_test))
    h_vec = spit_router.sidecar.read()   # 6-element health vector
    phi   = spit_router.pid.step(h_vec)  # CPU-aware PID

    for i in range(start, end):
        r = spit(phi, float(synth_retweets[i]), float(synth_followers[i]),
                 float(synth_burst[i]), synth_platform_enum,
                 all_token_ids[i], spit_config)
        spit_tau_vals.append(r["tau_dynamic"])
        spit_phi_vals.append(r["phi_physical"])
        spit_psi_vals.append(r["psi_socio"])
        spit_htoken_vals.append(r["h_token"])

# Capture one hardware snapshot for reporting
hw_snap = spit_router.sidecar.read_labeled()
spit_router.stop()

# Step 4: Escalation mask
esc_mask = [c < t for c, t in zip(t1_confs, spit_tau_vals)]
esc_idx  = [i for i, m in enumerate(esc_mask) if m]
esc_txts = [X_test[i] for i in esc_idx]
print(f"[SPIT] Escalating {len(esc_idx)}/{len(X_test)} ({len(esc_idx)/len(X_test):.1%}) to Tier-2 ...")

# Step 5: Batch Tier-2 for escalated samples
t2map={}; t2lat={}
if esc_txts:
    for s in range(0, len(esc_txts), INFERENCE_BATCH_SIZE):
        bt=esc_txts[s:s+INFERENCE_BATCH_SIZE]; bi=esc_idx[s:s+INFERENCE_BATCH_SIZE]
        t0=time.perf_counter(); pb,_=t2_predict_batch(bt); lat=(time.perf_counter()-t0)/len(bt)
        for idx, p in zip(bi, pb): t2map[idx]=p; t2lat[idx]=lat

# Step 6: Assemble predictions + latencies
spit_preds=[]; spit_latencies=[]; spit_tier2_count=len(esc_idx)
for i in range(len(X_test)):
    if i in t2map: spit_preds.append(t2map[i]); spit_latencies.append(t1_latencies[i]+t2lat[i])
    else:          spit_preds.append(t1_preds[i]); spit_latencies.append(t1_latencies[i])

# Step 7: Metrics
from sklearn.metrics import accuracy_score, f1_score
spit_acc = accuracy_score(y_test, spit_preds)
spit_f1  = f1_score(y_test, spit_preds, average="macro", zero_division=0)
spit_lat = float(np.mean(spit_latencies)) * 1000
spit_tier2_rate = spit_tier2_count / len(X_test)

print(f"\n{'='*58}")
print(f"  ASO+SPIT Results  |  Telemetry: {_TELEMETRY_MODE}")
print(f"{'='*58}")
print(f"  Accuracy     : {spit_acc:.4f}")
print(f"  F1 (Macro)   : {spit_f1:.4f}")
print(f"  Avg latency  : {spit_lat:.4f} ms/sample")
print(f"  Tier-2 rate  : {spit_tier2_count}/{len(X_test)} ({spit_tier2_rate:.1%})")
print(f"  tau range    : [{min(spit_tau_vals):.4f}, {max(spit_tau_vals):.4f}]")
print(f"  tau mean±std : {np.mean(spit_tau_vals):.4f} ± {np.std(spit_tau_vals):.4f}")
print(f"{'='*58}")
print(f"  Hardware telemetry snapshot (last batch):")
for k, v in hw_snap.items():
    note = ""
    if k == "cpu_temp_norm" and v == 0.0:
        note = "  ← unavailable on this VM (normal)"
    elif k in ("gpu_temp_norm", "vram_usage_norm") and v == 0.0 and not ON_GPU:
        note = "  ← no GPU"
    print(f"    {k:<22}: {v*100:5.1f} %{note}")
print(f"{'='*58}")


## Cell 8 — Results Table & Plots

In [ ]:
# ============================================================
# CELL 8: Final Comparison Table (5 Models)
# ============================================================
from sklearn.metrics import classification_report

results_data = [
    {"Model":"Logistic Regression (TF-IDF)",     "Accuracy":lr_acc,   "F1 Score":lr_f1,   "Avg Latency (ms)":lr_lat,  "Notes":"Classical baseline"},
    {"Model":"DistilBERT (Tier-1 only)",          "Accuracy":t1_acc,   "F1 Score":t1_f1,   "Avg Latency (ms)":t1_lat,  "Notes":"Fast transformer"},
    {"Model":"BERT (Tier-2 only)",                "Accuracy":t2_acc,   "F1 Score":t2_f1,   "Avg Latency (ms)":t2_lat,  "Notes":"Strong transformer"},
    {"Model":f"ASO (fixed tau={ASO_THRESHOLD})",  "Accuracy":aso_acc,  "F1 Score":aso_f1,  "Avg Latency (ms)":aso_lat, "Notes":f"Hybrid; {tier2_rate:.0%} T2"},
    {"Model":"ASO+SPIT (tau_dynamic)",            "Accuracy":spit_acc, "F1 Score":spit_f1, "Avg Latency (ms)":spit_lat,"Notes":f"SPIT; {spit_tier2_rate:.0%} T2"},
]

df=pd.DataFrame(results_data); dd=df.copy()
dd["Accuracy"]=df["Accuracy"].map("{:.4f}".format)
dd["F1 Score"]=df["F1 Score"].map("{:.4f}".format)
dd["Avg Latency (ms)"]=df["Avg Latency (ms)"].map("{:.4f}".format)

print("\n"+"="*90)
print("  ASO+SPIT PIPELINE -- FINAL RESULTS TABLE")
print("="*90)
print(dd.to_string(index=False))
print("="*90)

from IPython.display import display
styled=(dd.style
    .set_caption("Table 1: ASO+SPIT Pipeline -- Full Model Comparison")
    .set_table_styles([{"selector":"caption","props":"font-size:14px;font-weight:bold;text-align:left;"}])
    .highlight_max(subset=["Accuracy","F1 Score"],color="#d5f5d5")
    .highlight_min(subset=["Avg Latency (ms)"],color="#d5f5d5"))
display(styled)

In [ ]:
# ============================================================
# CELL 9: Bar Chart Comparison (5 Models)
# ============================================================
PALETTE     = ["#4C72B0","#DD8452","#55A868","#C44E52","#9467BD"]
short_names = ["LR","DistilBERT\n(T1)","BERT\n(T2)",f"ASO\n(t={ASO_THRESHOLD})","ASO\n+SPIT"]
accuracies  = [r["Accuracy"]         for r in results_data]
f1_scores   = [r["F1 Score"]         for r in results_data]
latencies   = [r["Avg Latency (ms)"] for r in results_data]

fig,axes=plt.subplots(1,3,figsize=(18,5))
fig.suptitle("ASO+SPIT Pipeline -- Model Comparison",fontsize=14,fontweight="bold")

def bar(ax,vals,title,ylabel,logy=False,fmt=".4f"):
    bs=ax.bar(range(len(short_names)),vals,color=PALETTE,width=0.55,edgecolor="white",linewidth=0.8)
    ax.set_xticks(range(len(short_names))); ax.set_xticklabels(short_names,ha="center",fontsize=9)
    ax.set_title(title,fontsize=12,fontweight="bold",pad=10); ax.set_ylabel(ylabel,fontsize=10)
    if logy: ax.set_yscale("log")
    for b,v in zip(bs,vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()*(1.04 if not logy else 1.2),
                f"{v:{fmt}}",ha="center",va="bottom",fontsize=8.5)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

bar(axes[0],accuracies,"(a) Accuracy","Accuracy",fmt=".4f")
bar(axes[1],f1_scores, "(b) F1 Score (Macro)","F1 Score",fmt=".4f")
bar(axes[2],latencies, "(c) Avg Latency (ms)","Latency (ms)",logy=True,fmt=".3f")
plt.tight_layout()
plt.savefig("aso_comparison_plot.png",dpi=150,bbox_inches="tight")
plt.show()
print("Saved: aso_comparison_plot.png")

In [ ]:
# ============================================================
# CELL 10: ASO Threshold Sweep (Ablation Study)
# Runs ASO at multiple τ values to show the accuracy-latency
# trade-off — key result for the paper.
# ============================================================

THRESHOLDS = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

sweep_rows = []
print(f"{'τ':>6} | {'Accuracy':>10} | {'F1 Macro':>10} | {'Avg Lat (ms)':>14} | {'Tier-2 (%)':>11}")
print("-" * 62)

for tau in THRESHOLDS:
    sweep_preds, sweep_lats = [], []
    t2_count = 0

    for start in range(0, len(X_test), INFERENCE_BATCH_SIZE):
        batch = X_test[start : start + INFERENCE_BATCH_SIZE]
        t1_bp, t1_bc = t1_predict_batch(batch)
        t1_time = 0.0  # already measured; use stored latencies for efficiency

        final = list(t1_bp)
        esc_idx   = [i for i, c in enumerate(t1_bc) if c < tau]
        esc_texts = [batch[i] for i in esc_idx]
        t2_count += len(esc_idx)

        if esc_texts:
            t2_bp, _ = t2_predict_batch(esc_texts)
            for rank, idx in enumerate(esc_idx):
                final[idx] = t2_bp[rank]

        sweep_preds.extend(final)

    s_acc  = accuracy_score(y_test, sweep_preds)
    s_f1   = f1_score(y_test, sweep_preds, average="macro", zero_division=0)
    t2_pct = t2_count / len(X_test) * 100
    # Approximate latency: weighted average of Tier-1 and Tier-2 latencies
    s_lat  = t1_lat * (1 - t2_pct/100) + t2_lat * (t2_pct/100)

    print(f"{tau:>6.2f} | {s_acc:>10.4f} | {s_f1:>10.4f} | {s_lat:>14.4f} | {t2_pct:>10.1f}%")
    sweep_rows.append({"Threshold": tau, "Accuracy": s_acc,
                       "F1 Macro": s_f1, "Avg Lat (ms)": s_lat,
                       "Tier-2 Rate (%)": t2_pct})

sweep_df = pd.DataFrame(sweep_rows)

# ── Plot the sweep ────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(sweep_df["Threshold"], sweep_df["Accuracy"],
         marker="o", color="#4C72B0", label="Accuracy", linewidth=2)
ax1.plot(sweep_df["Threshold"], sweep_df["F1 Macro"],
         marker="s", color="#55A868", label="F1 Macro", linewidth=2)
ax1.set_xlabel("Confidence Threshold (τ)", fontsize=11)
ax1.set_ylabel("Score", fontsize=11)
ax1.set_title("ASO Threshold Sweep — Accuracy & F1 vs. τ",
              fontsize=12, fontweight="bold")
ax1.set_ylim(0.5, 1.05)
ax1.spines["top"].set_visible(False)

ax2 = ax1.twinx()
ax2.plot(sweep_df["Threshold"], sweep_df["Tier-2 Rate (%)"],
         marker="^", color="#DD8452", linestyle="--",
         label="Tier-2 Rate (%)", linewidth=2)
ax2.set_ylabel("Tier-2 Invocation Rate (%)", color="#DD8452", fontsize=11)
ax2.tick_params(axis="y", labelcolor="#DD8452")
ax2.spines["top"].set_visible(False)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig("aso_threshold_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print("Threshold sweep plot saved: aso_threshold_sweep.png")

In [ ]:
# ============================================================
# CELL 11: Per-Class Classification Reports (All 5 Models)
# ============================================================
from sklearn.metrics import classification_report

target_names=["Negative","Positive"]
model_results=[
    ("Logistic Regression",          lr_preds),
    ("DistilBERT (Tier-1 only)",     t1_preds),
    ("BERT (Tier-2 only)",           t2_preds_all),
    (f"ASO (fixed tau={ASO_THRESHOLD})", aso_preds),
    ("ASO+SPIT (tau_dynamic)",       spit_preds),
]

for name,preds in model_results:
    print(f"\n{'?'*58}")
    print(f"  {name}")
    print(f"{'?'*58}")
    print(classification_report(y_test,preds,target_names=target_names,zero_division=0))

## Cell 12 ? SPIT Component Analysis & Ablation (6 Panels)

| Panel | Content |
|---|---|
| **(a) tau_dynamic dist.** | Histogram vs fixed tau=0.85 reference line |
| **(b) Phi_physical** | PID thermal signal across test set (shows GPU simulation) |
| **(c) Psi_socio dist.** | Socio-impact multiplier histogram |
| **(d) H_token dist.** | Linguistic entropy histogram |
| **(e) Tier-2 rate** | ASO fixed-tau vs ASO+SPIT side-by-side |
| **(f) tau vs Psi scatter** | Coloured by H_token -- shows joint signal interaction |

In [ ]:
# ============================================================
# CELL 13: SPIT Ablation -- Full Component Breakdown
# ============================================================
fig,axes=plt.subplots(2,3,figsize=(18,10))
fig.suptitle("SPIT Component Analysis -- Ablation Study",fontsize=15,fontweight="bold",y=1.01)

C1,C2,C3,C4="#9467BD","#E74C3C","#2ECC71","#3498DB"

# (a) tau_dynamic distribution
ax=axes[0,0]
ax.hist(spit_tau_vals,bins=30,color=C1,alpha=0.8,edgecolor="white")
ax.axvline(ASO_THRESHOLD,color=C2,linewidth=2,linestyle="--",label=f"Fixed tau={ASO_THRESHOLD}")
ax.set_xlabel("tau_dynamic"); ax.set_ylabel("Count")
ax.set_title("(a) tau_dynamic Distribution",fontweight="bold")
ax.legend(fontsize=9); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# (b) Phi_physical over samples
ax=axes[0,1]
ax.plot(spit_phi_vals,color=C2,linewidth=0.8,alpha=0.7)
ax.axhline(np.mean(spit_phi_vals),color="black",linewidth=1.2,linestyle="--",
           label=f"Mean={np.mean(spit_phi_vals):.4f}")
ax.set_xlabel("Sample Index"); ax.set_ylabel("Phi_physical")
ax.set_title("(b) Phi_physical -- PID Thermal Signal",fontweight="bold")
ax.legend(fontsize=9); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# (c) Psi_socio distribution
ax=axes[0,2]
ax.hist(spit_psi_vals,bins=30,color=C3,alpha=0.8,edgecolor="white")
ax.axvline(np.mean(spit_psi_vals),color="black",linewidth=1.2,linestyle="--",
           label=f"Mean={np.mean(spit_psi_vals):.4f}")
ax.set_xlabel("Psi_socio"); ax.set_ylabel("Count")
ax.set_title("(c) Psi_socio -- Socio-Impact Multiplier",fontweight="bold")
ax.legend(fontsize=9); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# (d) H_token distribution
ax=axes[1,0]
ax.hist(spit_htoken_vals,bins=30,color=C4,alpha=0.8,edgecolor="white")
ax.axvline(np.mean(spit_htoken_vals),color="black",linewidth=1.2,linestyle="--",
           label=f"Mean={np.mean(spit_htoken_vals):.4f}")
ax.set_xlabel("H_token"); ax.set_ylabel("Count")
ax.set_title("(d) H_token -- Linguistic Entropy",fontweight="bold")
ax.legend(fontsize=9); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# (e) Tier-2 rate comparison
ax=axes[1,1]
names_e=["ASO\n(fixed tau)","ASO+SPIT\n(dynamic tau)"]
rates_e=[tier2_rate*100, spit_tier2_rate*100]
bs=ax.bar(names_e,rates_e,color=["#DD8452",C1],width=0.45,edgecolor="white")
for b,r in zip(bs,rates_e):
    ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.5,f"{r:.1f}%",
            ha="center",va="bottom",fontsize=11,fontweight="bold")
ax.set_ylabel("Tier-2 Invocation Rate (%)")
ax.set_title("(e) Tier-2 Rate Comparison",fontweight="bold")
ax.set_ylim(0, max(rates_e)*1.3)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# (f) tau_dynamic vs Psi_socio coloured by H_token
ax=axes[1,2]
sc=ax.scatter(spit_psi_vals,spit_tau_vals,c=spit_htoken_vals,cmap="viridis",alpha=0.4,s=12)
plt.colorbar(sc,ax=ax,label="H_token")
ax.axhline(ASO_THRESHOLD,color=C2,linewidth=1.5,linestyle="--",alpha=0.7,label=f"Fixed tau={ASO_THRESHOLD}")
ax.set_xlabel("Psi_socio"); ax.set_ylabel("tau_dynamic")
ax.set_title("(f) tau_dynamic vs Psi_socio (colour=H_token)",fontweight="bold")
ax.legend(fontsize=8); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("spit_ablation_analysis.png",dpi=150,bbox_inches="tight")
plt.show()
print("Saved: spit_ablation_analysis.png")

# Component summary table
print("\n"+"="*58)
print("  SPIT Component Statistics")
print("="*58)
for nm,vals in [("Phi_physical",spit_phi_vals),("Psi_socio",spit_psi_vals),
                ("H_token",spit_htoken_vals),("tau_dynamic",spit_tau_vals)]:
    a=np.array(vals)
    print(f"  {nm:<14} mean={a.mean():.4f}  std={a.std():.4f}  range=[{a.min():.4f},{a.max():.4f}]")
print("="*58)
print(f"  Fixed ASO  Tier-2: {tier2_rate:.1%}")
print(f"  ASO+SPIT   Tier-2: {spit_tier2_rate:.1%}")
print(f"  Delta            : {(spit_tier2_rate-tier2_rate)*100:+.1f} pp")